# AKANE E2-5：聞き方を変えて、AIにルールを作らせ直す

前回（E2-4）は "Can something be both a mammal and a bird at the same time?" と聞いたら、Gemmaはほとんど「はい」と答えてしまった。
今回は聞き方を3通り用意し、**今回のテストに出てこない別の組**（両生類・甲殻類・チーズ・船など18個）で、どの聞き方が一番まともか先に決める。
決めた聞き方を、本番の分類テストに1回だけ使う（テストの正解を見て聞き方を選ばないため）。

- P1 "Can a mammal be a bird?"（先行研究で一番良かった形）
- P2 "Is it true that no mammal is a bird?"（逆向き）
- P0 前回と同じ形

**このノートでやること**：Gemmaに約2,200問（短い質問）を聞いて、答えをドライブに保存するだけ。そのあとの計算は手元のPCで行う。
**時間の目安**（T4 GPU）：5〜10分。計画は sims/e2_selfrule/PLAN_E2-5.md（結果を見る前に固定）。

## 0. 準備
1. 「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU**。
2. 鍵マークの `HF_TOKEN`（前と同じもの）にアクセスを許可。
3. 「ランタイム」→「すべてのセルを実行」。

In [ ]:
# ===== 設定 =====
USE_MOCK_MOUTH = False      # True: 作り物の口（動作確認用。結果は研究に使わない）
SAVE_TO_DRIVE  = True
MODEL_ID = "google/gemma-2-2b-it"
DATA_SEED = 0

In [ ]:
import os, json, time, itertools, collections
import numpy as np
for pkg, mod in [("python-sat", "pysat"), ("nltk", "nltk"), ("wordfreq", "wordfreq")]:
    try: __import__(mod)
    except ImportError: os.system(f"pip -q install {pkg}")
from pysat.formula import WCNF
from pysat.examples.rc2 import RC2
from sklearn.metrics import roc_auc_score

OUT = "/content/E2_results" if os.path.exists("/content") else "E2_results"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/AKANE_アーキテクチャ/E2_results"
os.makedirs(OUT, exist_ok=True)
print("保存先:", OUT)

## 1. テストを作る（前回と同じもの）

In [ ]:
import nltk
from nltk.corpus import wordnet as wn
from wordfreq import zipf_frequency
try: nltk.data.find("corpora/wordnet")
except LookupError:
    try: nltk.data.find("corpora/wordnet.zip")
    except LookupError: nltk.download("wordnet", quiet=True)
# 大分類（7つ）と種類（25）。種類は WordNet の見出し、英語の言い方、大分類
CONT = {"animal": "animal", "plant": "plant", "food": "food", "artifact": "man-made object",
        "activity": "activity", "condition": "medical condition", "feeling": "feeling"}
CATS = {"mammal.n.01": ("mammal", "animal"), "bird.n.01": ("bird", "animal"), "fish.n.01": ("fish", "animal"),
        "insect.n.01": ("insect", "animal"), "reptile.n.01": ("reptile", "animal"),
        "tree.n.01": ("tree", "plant"), "flower.n.01": ("flower", "plant"),
        "vegetable.n.01": ("vegetable", "food"), "edible_fruit.n.01": ("fruit", "food"), "beverage.n.01": ("drink", "food"),
        "dish.n.02": ("dish", "food"), "baked_goods.n.01": ("baked good", "food"),
        "tool.n.01": ("tool", "artifact"), "vehicle.n.01": ("vehicle", "artifact"),
        "musical_instrument.n.01": ("musical instrument", "artifact"), "weapon.n.01": ("weapon", "artifact"),
        "furniture.n.01": ("furniture", "artifact"), "garment.n.01": ("clothing", "artifact"),
        "container.n.01": ("container", "artifact"), "building.n.01": ("building", "artifact"),
        "sport.n.01": ("sport", "activity"), "dance.n.01": ("dance", "activity"), "game.n.01": ("game", "activity"),
        "disease.n.01": ("disease", "condition"), "emotion.n.01": ("emotion", "feeling")}
# どちらとも言える大分類の組（偽の問題に使わない）：植物と食べ物、動物と食べ物
AMBIG = {frozenset(("plant", "food")), frozenset(("animal", "food"))}
MASS = {"furniture", "clothing"}
art = lambda w: w if w in MASS else ("an " if w[0] in "aeiou" else "a ") + w
lemma = lambda s: s.lemmas()[0].name()
by_co = collections.defaultdict(list)          # 種類 → ことば
for sid in CATS:
    S = wn.synset(sid)
    for leaf in S.closure(lambda s: s.hyponyms()):
        w = lemma(leaf)
        if "_" in w or not w.isalpha() or not w.islower() or zipf_frequency(w, "en") < 2.5: continue
        if len(wn.synsets(w)) != 1: continue        # 意味が1つしかない語だけ（取り違えを避ける）
        anc = set(leaf.closure(lambda s: s.hypernyms()))
        hits = [c for c in CATS if wn.synset(c) in anc]
        if hits == [sid]: by_co[sid].append(w)
rng = np.random.default_rng(DATA_SEED)
COUNTRIES = [k for k in CATS if len(set(by_co[k])) >= 4]
STMT, GOLD, KIND, CITY = [], [], [], []
idx = {}
def add(key, text, gold, kind, city=None):
    if key in idx: return idx[key]
    idx[key] = len(STMT); STMT.append(text); GOLD.append(gold); KIND.append(kind); CITY.append(city)
    return idx[key]
cont_of = {k: CATS[k][1] for k in COUNTRIES}
same_cont = collections.defaultdict(list)
for k in COUNTRIES: same_cont[cont_of[k]].append(k)
others = lambda z: [w for w in CONT if w != z and frozenset((w, z)) not in AMBIG]
# 種類→大分類
for k in COUNTRIES:
    z = cont_of[k]; fz = rng.choice(others(z))
    for zz in (z, fz):
        add(("CK", k, zz), f"{art(CATS[k][0]).capitalize()} is a kind of {CONT[zz]}.", int(zz == z), "種類→大分類")
# ことば→種類、ことば→大分類（1つの種類から多くても30語）
CITIES = []
for k in COUNTRIES:
    ws = sorted(set(by_co[k]))
    pick = list(rng.choice(ws, min(30, len(ws)), replace=False))
    for name in pick:
        z = cont_of[k]; CITIES.append(name)
        same = [w for w in same_cont[z] if w != k]     # 偽の種類は、なるべく同じ大分類から2つ
        pool = same if len(same) >= 2 else same + [w for w in COUNTRIES if cont_of[w] in others(z)]
        fks = list(rng.choice(pool, 2, replace=False))
        for kk in [k] + fks:
            add(("CC", name, kk), f"{name.capitalize()} is a kind of {CATS[kk][0]}.", int(kk == k), "ことば→種類", name)
        for zz in (z, rng.choice(others(z))):
            add(("CL", name, zz), f"{name.capitalize()} is a kind of {CONT[zz]}.", int(zz == z), "ことば→大分類", name)
GOLD = np.array(GOLD)
DUMMY = len(STMT)   # 「両立しない」ルールの結論に使う、いつも0の素子
# ルール：(前提1, 前提2, 結論)。結論がDUMMYなら「前提が両方1ではいけない」
RULES = set()
for (t, a, b), i in list(idx.items()):
    if t == "CC":
        for zz in CONT:
            if ("CK", b, zz) in idx and ("CL", a, zz) in idx:
                RULES.add((i, idx[("CK", b, zz)], idx[("CL", a, zz)]))     # ことばが種類に入り、種類が大分類に入る → ことばはその大分類
def excl(keys):
    ids = [idx[k] for k in keys if k in idx]
    for p, q in itertools.combinations(sorted(ids), 2): RULES.add((p, q, DUMMY))
for name in set(CITIES):
    excl([("CC", name, k) for k in COUNTRIES]); excl([("CL", name, z) for z in CONT])
for k in COUNTRIES:
    excl([("CK", k, z) for z in CONT])
RULES = sorted(RULES)
# ルールが正解で全部守られているか（守られていなければデータの誤り）
G = np.append(GOLD, 0)
bad = sum(1 for p, q, h in RULES if G[p] == 1 and G[q] == 1 and G[h] == 0)
print(f"種類 {len(COUNTRIES)}、ことば {len(set(CITIES))}、問題 {len(STMT)}（本物 {GOLD.sum()}）、ルール {len(RULES)}本、正解が破るルール {bad}")
print("例:", " | ".join(STMT[i] for i in [0, 1, 60, 61, 62, 63, 64]))

## 2. 本番の組（前回と同じ314組）と、聞き方を決めるための別の組（153組）

In [ ]:
PAIRS = set()
for name in set(CITIES):
    L = [("K", k) for k in COUNTRIES if ("CC", name, k) in idx] + [("Z", z) for z in CONT if ("CL", name, z) in idx]
    for p, q in itertools.combinations(L, 2): PAIRS.add(tuple(sorted((p, q))))
for k in COUNTRIES:
    for p, q in itertools.combinations([("Z", z) for z in CONT if ("CK", k, z) in idx], 2): PAIRS.add(tuple(sorted((p, q))))
PAIRS = sorted(PAIRS)
def truth_excl(p, q):                      # True = 両立しない（WordNet）
    if p[0] == q[0]: return True
    k, z = (p, q) if p[0] == "K" else (q, p)
    return cont_of[k[1]] != z[1]
MASS2 = {"furniture", "clothing", "food"}
def name_of(lab):
    w = CATS[lab[1]][0] if lab[0] == "K" else CONT[lab[1]]
    return w if w in MASS2 else ("an " if w[0] in "aeiou" else "a ") + w

TEST = [(name_of(p), name_of(q), truth_excl(p, q)) for p, q in PAIRS]
# 聞き方を決める組（テストに出てこないラベル。正解は人が常識で付けた）
DEV_KIND = ["amphibian", "crustacean", "spider", "shrub", "grass", "mushroom", "cheese", "candy", "ship", "aircraft", "painting", "metal"]
DEV_TOP = ["vertebrate", "invertebrate", "living thing", "dairy product", "work of art", "means of transport"]
DEV_COMPAT = {("amphibian", "vertebrate"), ("amphibian", "living thing"), ("crustacean", "invertebrate"), ("crustacean", "living thing"),
              ("spider", "invertebrate"), ("spider", "living thing"), ("shrub", "living thing"), ("grass", "living thing"),
              ("mushroom", "living thing"), ("vertebrate", "living thing"), ("invertebrate", "living thing"),
              ("cheese", "dairy product"), ("ship", "means of transport"), ("aircraft", "means of transport"), ("painting", "work of art")}
DEV_AMBIG = {("metal", "work of art")}
MASS3 = {"cheese", "candy", "metal", "grass", "jewelry", "furniture", "clothing", "food"}
a_ = lambda w: w if w in MASS3 else ("an " if w[0] in "aeiou" else "a ") + w
DEV = []
for x, y in itertools.combinations(DEV_KIND + DEV_TOP, 2):
    if (x, y) in DEV_AMBIG or (y, x) in DEV_AMBIG: continue
    DEV.append((a_(x), a_(y), not ((x, y) in DEV_COMPAT or (y, x) in DEV_COMPAT)))
bare = lambda w: w.split(" ", 1)[1] if w.startswith(("a ", "an ")) else w
FORMS = {"P0": lambda x, y: f"Can something be both {x} and {y} at the same time? Answer Yes or No only.",
         "P1": lambda x, y: f"Can {x} be {y}? Answer Yes or No only.",
         "P2": lambda x, y: f"Is it true that no {bare(x)} is {y}? Answer Yes or No only."}
YES_MEANS_EXCL = {"P0": False, "P1": False, "P2": True}
QS = []   # (組の種類, 番号, 聞き方, 順番, 質問)
for setname, items in [("dev", DEV), ("test", TEST)]:
    for i, (x, y, _) in enumerate(items):
        for f, fn in FORMS.items():
            if setname == "test" and f == "P0": continue     # 本番のP0は前回（E2-4）の答えを使う
            QS.append((setname, i, f, 0, fn(x, y))); QS.append((setname, i, f, 1, fn(y, x)))
print(f"本番の組 {len(TEST)}、決め用の組 {len(DEV)}（両立する {sum(not t for *_, t in DEV)}）、質問 {len(QS)}")
for f, fn in FORMS.items(): print(f, fn("a mammal", "a bird"))

## 3. Gemmaに聞く（保存するので2回目からは飛ばせる）

In [ ]:
RQ = [q[-1] for q in QS]
rcache = f"{OUT}/E2-5_rules_{'MOCK' if USE_MOCK_MOUTH else MODEL_ID.split('/')[-1]}.json"
if os.path.exists(rcache):
    d = json.load(open(rcache))
    if d["questions"] != RQ: raise SystemExit("保存済みの答えと質問が合わない。ファイルを消してやり直す")
    RLOGIT = np.array(d["logit"]); print("保存済みの答えを読み込んだ:", rcache)
elif USE_MOCK_MOUTH:
    r = np.random.default_rng(7)
    tr = np.array([(DEV if s == "dev" else TEST)[i][2] for s, i, f, o, q in QS])
    sign = np.array([1 if YES_MEANS_EXCL[f] else -1 for s, i, f, o, q in QS])
    RLOGIT = sign * np.where(tr, 2.0, -2.0) + r.normal(0, 2.0, len(QS))
else:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from google.colab import userdata
    if "model" not in globals():
        tok = AutoTokenizer.from_pretrained(MODEL_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=userdata.get("HF_TOKEN"),
                                                     torch_dtype=torch.float16, device_map="cuda").eval()
    YES = tok.encode("Yes", add_special_tokens=False)[0]; NO = tok.encode("No", add_special_tokens=False)[0]
    def rprompt(s):
        return tok.apply_chat_template([{"role": "user", "content": s}], tokenize=False, add_generation_prompt=True)
    RLOGIT = np.zeros(len(RQ)); B = 64; t = time.time()
    for i in range(0, len(RQ), B):
        enc = tok([rprompt(s) for s in RQ[i:i + B]], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            last = model(**enc).logits[:, -1, :].float()
        RLOGIT[i:i + B] = (last[:, YES] - last[:, NO]).cpu().numpy()
        if (i // B) % 10 == 0: print(f"{i + B}/{len(RQ)}  {time.time() - t:.0f}秒")
if not os.path.exists(rcache):
    json.dump({"model": "MOCK" if USE_MOCK_MOUTH else MODEL_ID, "qs": QS, "questions": RQ, "logit": RLOGIT.tolist(),
               "test_pairs": PAIRS, "test_truth": [t for *_, t in TEST], "dev": DEV}, open(rcache, "w"), ensure_ascii=False)
    print("保存:", rcache)

## 4. 別の組で聞き方を決める（テストの正解は使わない）→ 本番での当たり具合

In [ ]:
def excl_of(setname, form):
    n = len(DEV if setname == "dev" else TEST); v = np.zeros((n, 2))
    for (s, i, f, o, q), l in zip(QS, RLOGIT):
        if s == setname and f == form: v[i, o] = l
    a = v.mean(1)
    return (a > 0) if YES_MEANS_EXCL[form] else (a < 0)
def bal(pred, truth):
    t = np.array(truth); return 0.5 * ((pred[t]).mean() + (~pred[~t]).mean())
dt = [t for *_, t in DEV]
dev_pred = {f: excl_of("dev", f) for f in FORMS}; dev_pred["P12"] = dev_pred["P1"] & dev_pred["P2"]
score = {f: bal(p, dt) for f, p in dev_pred.items()}
for f, s_ in score.items(): print(f"決め用の組 {f}: 釣り合い正解率 {s_:.3f}（両立しない組 {dev_pred[f][np.array(dt)].mean():.2f}、両立する組 {(~dev_pred[f][~np.array(dt)]).mean():.2f}）")
best = max(["P1", "P0", "P2", "P12"], key=lambda f: score[f])
print("→ 本番に使う聞き方:", best)
if best != "P0":
    tt = np.array([t for *_, t in TEST])
    tp = {f: excl_of("test", f) for f in ["P1", "P2"]}; tp["P12"] = tp["P1"] & tp["P2"]
    g = tp[best]
    print(f"本番の314組：正しい判断 {(g == tt).mean():.1%}、見逃し {(tt & ~g).sum()}/{tt.sum()}、間違ったルール {(~tt & g).sum()}/{(~tt).sum()}")